# Diagnóstico de calidad de datos: propiedad horizontal en Medellín

**Evidencia:** GA1-220501094-01-AA1-EV06 – Diagnosticar la calidad de datos: preparación, exploración y documentación

**Aprendiz:** Maicol Esneider Posada Callejas · **Ficha:** 3223873

**Programa:** Tecnólogo en Análisis y Desarrollo de Software (228118) · SENA

**Instructor:** Carlos Navia · **Fecha:** noviembre de 2026

---

**Contenido**

1. Reconocimiento del proyecto y de los datasets
2. Preparación del entorno y carga de datos
3. Inspección de la estructura de los datasets
4. Diagnóstico inicial de calidad

El informe técnico completo está en `reports/informe_diagnostico.md`.

## Momento 1. Reconocimiento del proyecto y de los datasets

**Nombre del proyecto:** Distribución de la propiedad horizontal en las comunas de Medellín.

**Propósito:** saber en qué zonas de Medellín hay más predios en propiedad horizontal (conjuntos residenciales y edificios de apartamentos) para decidir dónde enfocar primero Domux, un software de administración de propiedades horizontales que se desarrolla como proyecto de grado.

**Pregunta analítica (formulada en la EV02):**
> ¿En qué comunas de Medellín se concentran los predios habitacionales en propiedad horizontal y qué proporción representan frente al total de predios habitacionales de cada comuna?

### Datasets seleccionados

| N.º | Dataset (archivo) | Fuente | Qué contiene | Relación con el problema |
|---|---|---|---|---|
| 1 | Información Predios (`informacion_predios.csv`) | MEData – Subsecretaría de Catastro de Medellín | Un registro por predio: comuna, barrio, manzana, lote (CBML), matrícula, porcentaje de copropiedad, estrato, uso predial y avalúo. | Es la fuente principal: permite contar predios por comuna y estimar cuáles están en propiedad horizontal. |
| 2 | Información Lotes (`informacion_lotes.csv`) | MEData – Subsecretaría de Catastro de Medellín | Un registro por lote: área del lote, área construida, zona geoeconómica y número de pisos. | Complementa los predios: un lote con muchos pisos y muchos predios indica un edificio o conjunto. |
| 3 | Medellín Properties 2023 (`medellin_properties.csv`) | Kaggle (usuario cesaregr) | Inmuebles en venta en 2023: barrio, coordenadas, tipo, precio, habitaciones, área, valor de administración, estrato. | Muestra la oferta de inmuebles por barrio y el cobro de administración, típico de la propiedad horizontal. |
| 4 | Precios inmuebles Medellín (`houses_medellin.csv`) | Kaggle (usuario valentinafeve) | Inmuebles en venta: habitaciones, baños, estrato, antigüedad, administración, precio y área. | Sirve para relacionar el valor de la administración con el estrato y el área. |

Los conjuntos 1 y 2 se identificaron en la EV02 y la EV03; los conjuntos 3 y 4, en la EV03. Junto a los CSV de MEData se guardaron sus diccionarios de datos (`Informacion_Predios.json` e `Informacion_Lotes.json`).

### Aspectos de calidad que necesito revisar

1. **Completitud:** columnas con valores vacíos o con valores de relleno (por ejemplo, "--" o "No definida").
2. **Tipos de datos:** que los números estén como números y los códigos como texto (por ejemplo, el código CBML tiene ceros a la izquierda).
3. **Coherencia:** valores imposibles o sospechosos (áreas en 0, coordenadas fuera de Medellín, precios extremos, duplicados).
4. **Correspondencia con el diccionario:** que los nombres de las columnas coincidan con la documentación.
5. **Utilidad para la pregunta:** si los datos permiten identificar la propiedad horizontal y ubicarla por comuna.

## Momento 2. Preparación del entorno y carga de datos

El entorno se prepara siguiendo la guía *Proyecto práctica: entorno y cuaderno* (entorno virtual `.venv`, dependencias en `requirements.txt` y JupyterLab). Las instrucciones están en el `README.md`.

Los cuatro archivos son CSV separados por coma, así que se cargan con `pd.read_csv()`. Cada uno queda en un DataFrame con un nombre que lo identifica.

In [1]:
import warnings
import pandas as pd

# pandas 3 muestra un aviso de compatibilidad al usar describe(include='object');
# se oculta solo ese aviso para que la salida sea más fácil de leer.
warnings.filterwarnings("ignore", message="For backward compatibility, 'str' dtypes")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Carpeta de los datos, relativa a la carpeta notebooks/
RUTA_DATOS = "../data/raw/"

df_predios = pd.read_csv(RUTA_DATOS + "informacion_predios.csv")
df_lotes = pd.read_csv(RUTA_DATOS + "informacion_lotes.csv")
df_propiedades = pd.read_csv(RUTA_DATOS + "medellin_properties.csv")
df_inmuebles = pd.read_csv(RUTA_DATOS + "houses_medellin.csv")

print("Versión de pandas:", pd.__version__)

Versión de pandas: 3.0.5


**Verificación de la carga:** si un archivo se leyera con el separador equivocado, quedaría con una sola columna. Compruebo que cada DataFrame tenga filas y varias columnas.

In [2]:
datasets = {
    "Información Predios": df_predios,
    "Información Lotes": df_lotes,
    "Medellín Properties 2023": df_propiedades,
    "Precios inmuebles Medellín": df_inmuebles,
}

for nombre, df in datasets.items():
    estado = "OK" if df.shape[0] > 0 and df.shape[1] > 1 else "REVISAR"
    print(f"{nombre:28} {df.shape[0]:>9} filas  {df.shape[1]:>3} columnas  -> {estado}")

Información Predios            1041413 filas   15 columnas  -> OK
Información Lotes               308129 filas    8 columnas  -> OK
Medellín Properties 2023          9999 filas   12 columnas  -> OK
Precios inmuebles Medellín         430 filas   14 columnas  -> OK


## Momento 3. Inspección de la estructura de los datasets

Para cada DataFrame ejecuto `head()`, `tail()`, `shape`, `columns`, `dtypes`, `info()`, `describe()` y `describe(include='object')`.

### 3.1 Información Predios

In [3]:
df_predios.head()

,codComuna,codBarrio,Manzana,CedCat,CBML,Matricula,PorDes,Estrato,Urbanizacion,UsoPredial,IndicadorU/R,NroPropietarios,nomComuna,DS_BARRIO,AvaluoTotal
0,11,5,55,1,11050550001,257792,0.085,0,TORRES DE RIO NO 1 Y 2,10,U,0,LAURELES,LOS CONQUISTADORES,17500000.0
1,11,5,55,1,11050550001,257821,0.085,0,TORRES DE RIO NO 1 Y 2,10,U,0,LAURELES,LOS CONQUISTADORES,17500000.0
2,11,9,28,27,11090280027,258113,33.823,5,--,1,U,0,LAURELES,LAS ACACIAS,326925000.0
3,80,98,11,15,80980110015,1245992,0.020,0,RESERVAS DEL PRADO ETAPA 2,10,U,0,SAN ANTONIO DE PRADO,ZONA EXPANSION SAN ANTONIO,6342000.0
4,80,98,11,15,80980110015,1246000,0.020,0,RESERVAS DEL PRADO ETAPA 2,10,U,0,SAN ANTONIO DE PRADO,ZONA EXPANSION SAN ANTONIO,6342000.0


In [4]:
df_predios.tail()

,codComuna,codBarrio,Manzana,CedCat,CBML,Matricula,PorDes,Estrato,Urbanizacion,UsoPredial,IndicadorU/R,NroPropietarios,nomComuna,DS_BARRIO,AvaluoTotal
1041408,14,4,1,166,14040010166,1239991,0.005,0,FONT,10,U,0,EL POBLADO,CASTROPOL,3570000.0
1041409,14,4,1,166,14040010166,1239996,0.005,0,FONT,10,U,0,EL POBLADO,CASTROPOL,3570000.0
1041410,14,4,1,166,14040010166,1239999,0.004,0,FONT,10,U,0,EL POBLADO,CASTROPOL,2380000.0
1041411,14,4,1,166,14040010166,1240021,0.022,0,FONT,10,U,0,EL POBLADO,CASTROPOL,17500000.0
1041412,14,4,1,166,14040010166,1240023,0.011,0,FONT,10,U,0,EL POBLADO,CASTROPOL,7140000.0


In [5]:
print("shape:", df_predios.shape)
print("\ncolumns:", list(df_predios.columns))
print("\ndtypes:")
print(df_predios.dtypes)

shape: (1041413, 15)

columns: ['codComuna', 'codBarrio', 'Manzana', 'CedCat', 'CBML', 'Matricula', 'PorDes', 'Estrato', 'Urbanizacion', 'UsoPredial', 'IndicadorU/R', 'NroPropietarios', 'nomComuna', 'DS_BARRIO', 'AvaluoTotal']

dtypes:
codComuna            int64
codBarrio            int64
Manzana              int64
CedCat               int64
CBML                 int64
Matricula            int64
PorDes             float64
Estrato              int64
Urbanizacion           str
UsoPredial           int64
IndicadorU/R           str
NroPropietarios      int64
nomComuna              str
DS_BARRIO              str
AvaluoTotal        float64
dtype: object


In [6]:
df_predios.info()

<class 'pandas.DataFrame'>
RangeIndex: 1041413 entries, 0 to 1041412
Data columns (total 15 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   codComuna        1041413 non-null  int64  
 1   codBarrio        1041413 non-null  int64  
 2   Manzana          1041413 non-null  int64  
 3   CedCat           1041413 non-null  int64  
 4   CBML             1041413 non-null  int64  
 5   Matricula        1041413 non-null  int64  
 6   PorDes           1041413 non-null  float64
 7   Estrato          1041413 non-null  int64  
 8   Urbanizacion     1035613 non-null  str    
 9   UsoPredial       1041413 non-null  int64  
 10  IndicadorU/R     1041413 non-null  str    
 11  NroPropietarios  1041413 non-null  int64  
 12  nomComuna        1041413 non-null  str    
 13  DS_BARRIO        1041413 non-null  str    
 14  AvaluoTotal      1041413 non-null  float64
dtypes: float64(2), int64(9), str(4)
memory usage: 119.2 MB


In [7]:
df_predios.describe()

,codComuna,codBarrio,Manzana,CedCat,CBML,Matricula,PorDes,Estrato,UsoPredial,NroPropietarios,AvaluoTotal
count,1.041413e+06,1.041413e+06,1.041413e+06,1.041413e+06,1.041413e+06,1.041413e+06,1.041413e+06,1.041413e+06,1.041413e+06,1041413.0,1.041413e+06
mean,1.556986e+01,1.311179e+01,3.204838e+01,5.653089e+01,1.570130e+10,6.267931e+07,2.959330e+01,1.916697e+00,3.429216e+00,0.0,1.081875e+08
std,1.762161e+01,1.853113e+01,8.750137e+01,1.714224e+02,1.771913e+10,2.219096e+08,3.947600e+01,1.825391e+00,3.797826e+00,0.0,1.209903e+09
min,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.010010e+09,2.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.0,2.800000e+04
25%,8.000000e+00,4.000000e+00,5.000000e+00,7.000000e+00,8.010050e+09,5.453320e+05,2.300000e-01,0.000000e+00,1.000000e+00,0.0,1.617300e+07
50%,1.100000e+01,9.000000e+00,1.500000e+01,1.800000e+01,1.115016e+10,9.872820e+05,3.434000e+00,2.000000e+00,1.000000e+00,0.0,3.302900e+07
75%,1.400000e+01,1.400000e+01,3.200000e+01,4.500000e+01,1.421030e+10,5.150826e+06,5.000000e+01,3.000000e+00,8.000000e+00,0.0,8.327300e+07
max,9.000000e+01,9.800000e+01,9.990000e+02,3.222000e+03,9.089001e+10,9.601176e+08,1.000000e+02,6.000000e+00,1.700000e+01,0.0,6.249870e+11


In [8]:
df_predios.describe(include='object')

,Urbanizacion,IndicadorU/R,nomComuna,DS_BARRIO
count,1035613,1041413,1041413,1041413
unique,10935,2,21,348
top,--,U,EL POBLADO,CABECERA SAN ANTONIO DE PRADO
freq,590022,1021697,174543,25654


**Características principales de Información Predios**

- Tiene **1.041.413 predios** y **15 columnas**. Es el dataset más grande del proyecto.
- 11 columnas son numéricas y 4 de texto (`Urbanizacion`, `IndicadorU/R`, `nomComuna`, `DS_BARRIO`).
- `CBML` y `Matricula` son **códigos**, pero pandas los leyó como números (`int64`). Por eso `describe()` les calcula promedio y desviación, que no tienen sentido.
- `NroPropietarios` tiene media 0 y desviación 0: **todos sus valores son 0**.
- `Estrato` va de 0 a 6; el 0 no es un estrato socioeconómico real.
- `PorDes` (porcentaje de copropiedad) va de 0 a 100, con mediana 3,43: la mayoría de los predios comparten el lote con otros.
- `AvaluoTotal` tiene un máximo de más de 600.000 millones de pesos, muy lejos de la mediana (unos 33 millones).
- En las columnas de texto hay 21 comunas y corregimientos, 348 barrios y 590.022 predios con `Urbanizacion` igual a "--".

### 3.2 Información Lotes

In [9]:
df_lotes.head()

,CBML,codComuna,nomComuna,AreaLote,AreaCons,Zona,ZonaGeo,numPisos
0,1010020003,1,POPULAR,33.0,85.0,440,1.221214e+14,2
1,1010020010,1,POPULAR,145.0,188.0,440,1.221214e+14,2
2,1010020014,1,POPULAR,55.0,173.0,440,2.031214e+14,3
3,1010030013,1,POPULAR,123.0,102.0,440,1.111314e+14,2
4,1010030021,1,POPULAR,51.0,129.0,440,1.111314e+14,2


In [10]:
df_lotes.tail()

,CBML,codComuna,nomComuna,AreaLote,AreaCons,Zona,ZonaGeo,numPisos
308124,90880010011,90,SANTA ELENA,6542.0,93.0,330,7.225123e+14,1
308125,90890010010,90,SANTA ELENA,5952.0,0.0,330,7.225138e+14,0
308126,90890010014,90,SANTA ELENA,11628.0,0.0,330,8.045138e+14,0
308127,90890010028,90,SANTA ELENA,15555.0,81.0,349,7.225144e+14,1
308128,90890010035,90,SANTA ELENA,1029.0,87.0,349,7.225144e+14,2


In [11]:
print("shape:", df_lotes.shape)
print("\ncolumns:", list(df_lotes.columns))
print("\ndtypes:")
print(df_lotes.dtypes)

shape: (308129, 8)

columns: ['CBML', 'codComuna', 'nomComuna', 'AreaLote', 'AreaCons', 'Zona', 'ZonaGeo', 'numPisos']

dtypes:
CBML           int64
codComuna      int64
nomComuna        str
AreaLote     float64
AreaCons     float64
Zona           int64
ZonaGeo      float64
numPisos       int64
dtype: object


In [12]:
df_lotes.info()

<class 'pandas.DataFrame'>
RangeIndex: 308129 entries, 0 to 308128
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   CBML       308129 non-null  int64  
 1   codComuna  308129 non-null  int64  
 2   nomComuna  308129 non-null  str    
 3   AreaLote   308129 non-null  float64
 4   AreaCons   308129 non-null  float64
 5   Zona       308129 non-null  int64  
 6   ZonaGeo    308129 non-null  float64
 7   numPisos   308129 non-null  int64  
dtypes: float64(3), int64(4), str(1)
memory usage: 18.8 MB


In [13]:
df_lotes.describe()

,CBML,codComuna,AreaLote,AreaCons,Zona,ZonaGeo,numPisos
count,3.081290e+05,308129.000000,3.081290e+05,308129.000000,308129.000000,3.081290e+05,308129.000000
mean,1.508506e+10,14.976941,1.146882e+03,310.621834,439.901872,1.580256e+14,2.181736
std,2.037203e+10,20.306373,2.979675e+04,1825.444392,83.673147,1.267705e+14,1.832560
min,1.010010e+09,1.000000,0.000000e+00,0.000000,30.000000,1.960000e+02,0.000000
25%,5.040250e+09,5.000000,5.600000e+01,58.000000,435.000000,1.111215e+14,1.000000
50%,9.030710e+09,9.000000,9.600000e+01,139.000000,455.000000,1.111325e+14,2.000000
75%,1.501008e+10,15.000000,1.730000e+02,246.000000,480.000000,1.421205e+14,3.000000
max,9.089001e+10,90.000000,1.113594e+07,209310.000000,600.000000,8.058149e+14,39.000000


In [14]:
df_lotes.describe(include='object')

,nomComuna
count,308129
unique,21
top,GUAYABAL
freq,29596


**Características principales de Información Lotes**

- Tiene **308.129 lotes** y **8 columnas**, sin valores nulos.
- Igual que en predios, `CBML` se leyó como número entero.
- `ZonaGeo` es un **código** de 15 dígitos, pero quedó como decimal (`float64`) y se muestra en notación científica.
- `AreaLote` tiene un máximo de 11.135.942 m², muy por encima de la mediana (96 m²); son lotes rurales muy grandes.
- `AreaCons` y `numPisos` tienen mínimo 0: hay lotes sin construcción registrada.
- `numPisos` llega hasta 39 pisos, lo que coincide con edificios altos.

### 3.3 Medellín Properties 2023

In [15]:
df_propiedades.head()

,neighbourhood,latitude,longitude,property_type,price,rooms,baths,area,administration_price,age,garages,stratum
0,Suramerica,6.1862025,-75.5994371,Apartamento,435000000,3.0,2.0,83.0,354400.0,1.0,1,4
1,Escobero,6.1627997,-75.57351899999999,Apartamento,680000000,3.0,4.0,124.0,480000.0,2.0,2,4
2,Castropol,6.2161401,-75.56697,Apartamento,900000000,3.0,3.0,111.0,813000.0,1.0,2,6
3,Toledo,6.1627622,-75.6393071,Casa,650000000,3.0,2.0,127.0,0.0,2.0,1,4
4,La pilarica,6.2476376,-75.56581530000001,Apartamento,320000000,3.0,2.0,72.0,250000.0,NaN,2,5


In [16]:
df_propiedades.tail()

,neighbourhood,latitude,longitude,property_type,price,rooms,baths,area,administration_price,age,garages,stratum
9994,Loma de los Bernal,6.213318,-75.60780559999999,Apartamento,685000000,3.0,3.0,94.0,380000.0,1.0,2,5
9995,Conjunto residencial san jose,6.1573101,-75.5782206,Apartamento,582000000,3.0,2.0,88.0,0.0,2.0,1,4
9996,Centro,6.2447317,-75.5607505,Apartamento,320000000,3.0,2.0,59.0,0.0,NaN,NaN,4
9997,La america,6.2544524,-75.6113306,Apartamento,450000000,4.0,3.0,147.0,0.0,3.0,1,3
9998,La america,6.2509783,-75.60458919999999,Apartamento,245000000,2.0,2.0,51.0,50000.0,2.0,NaN,4


In [17]:
print("shape:", df_propiedades.shape)
print("\ncolumns:", list(df_propiedades.columns))
print("\ndtypes:")
print(df_propiedades.dtypes)

shape: (9999, 12)

columns: ['neighbourhood', 'latitude', 'longitude', 'property_type', 'price', 'rooms', 'baths', 'area', 'administration_price', 'age', 'garages', 'stratum']

dtypes:
neighbourhood               str
latitude                    str
longitude                   str
property_type               str
price                     int64
rooms                   float64
baths                   float64
area                    float64
administration_price    float64
age                     float64
garages                     str
stratum                   int64
dtype: object


In [18]:
df_propiedades.info()

<class 'pandas.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   neighbourhood         9999 non-null   str    
 1   latitude              9999 non-null   str    
 2   longitude             9999 non-null   str    
 3   property_type         9999 non-null   str    
 4   price                 9999 non-null   int64  
 5   rooms                 9936 non-null   float64
 6   baths                 9877 non-null   float64
 7   area                  9999 non-null   float64
 8   administration_price  9999 non-null   float64
 9   age                   7362 non-null   float64
 10  garages               6793 non-null   str    
 11  stratum               9999 non-null   int64  
dtypes: float64(5), int64(2), str(5)
memory usage: 937.5 KB


In [19]:
df_propiedades.describe()

,price,rooms,baths,area,administration_price,age,stratum
count,9.999000e+03,9936.000000,9877.000000,9.999000e+03,9.999000e+03,7362.00000,9999.000000
mean,1.451617e+09,3.008253,2.452060,3.933636e+03,1.217442e+06,1.76270,4.058806
std,3.725750e+10,1.020047,1.105903,2.735696e+05,2.302714e+07,1.10306,1.036415
min,1.600000e+06,1.000000,1.000000,0.000000e+00,0.000000e+00,0.00000,0.000000
25%,3.000000e+08,3.000000,2.000000,6.400000e+01,0.000000e+00,1.00000,3.000000
50%,4.200000e+08,3.000000,2.000000,8.400000e+01,1.580000e+05,1.00000,4.000000
75%,6.650000e+08,3.000000,3.000000,1.300000e+02,3.200000e+05,3.00000,5.000000
max,3.100000e+12,20.000000,10.000000,2.350000e+07,1.488000e+09,4.00000,6.000000


In [20]:
df_propiedades.describe(include='object')

,neighbourhood,latitude,longitude,property_type,garages
count,9999,9999,9999,9999,6793
unique,1320,4577,4476,4,11
top,Calasanz,6.167286499999999,-75.5836963,Apartamento,1
freq,337,517,517,7973,4655


**Características principales de Medellín Properties 2023**

- Tiene **9.999 inmuebles** y **12 columnas**.
- `latitude` y `longitude` deberían ser decimales, pero quedaron como **texto**: algunas filas tienen la palabra "INVALID".
- `garages` también quedó como texto porque incluye el valor "Más de 10".
- `rooms`, `baths`, `age` y `garages` tienen valores vacíos (`count` menor que el total de filas).
- `price` va de 1,6 millones a 3,1 billones de pesos; `area` va de 0 a 23,5 millones de m². Hay valores extremos imposibles para una vivienda.
- `age` no son años, sino un código de 0 a 4 (rango de antigüedad), como lo dice la descripción del dataset.
- Hay 1.320 barrios distintos y 4 tipos de inmueble (Apartamento, Casa, Apartaestudio, Proyecto).

### 3.4 Precios inmuebles Medellín

In [21]:
df_inmuebles.head()

,Unnamed: 0,Habitaciones,Baños,Estrato,Antigüedad,Piso N°,Administración,Precio m²,Parqueaderos,Estado,Tipo de apartamento,Precio,Área construída (m²),Área privada (m²)
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,240000000,NaN,NaN
1,1,1.0,2.0,4.0,1 a 8 años,3,No definida,"$ 4.032.258,06*m²",NaN,NaN,NaN,250000000,62,62
2,2,1.0,2.0,6.0,9 a 15 años,NaN,No definida,"$ 6.428.571,43*m²",2.0,NaN,NaN,675000000,105,0
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,647307563,NaN,NaN
4,4,1.0,1.0,5.0,NaN,3,$ 167.000 COP,$ 4.700.000*m²,2.0,NaN,NaN,235000000,50,0


In [22]:
df_inmuebles.tail()

,Unnamed: 0,Habitaciones,Baños,Estrato,Antigüedad,Piso N°,Administración,Precio m²,Parqueaderos,Estado,Tipo de apartamento,Precio,Área construída (m²),Área privada (m²)
425,425,3.0,1.0,3.0,1 a 8 años,16,$ 160.000 COP,"$ 3.541.666,67*m²",1.0,Excelente,NaN,170000000,48,48
426,426,3.0,3.0,4.0,9 a 15 años,5,$ 300.000 COP,"$ 4.429.347,83*m²",1.0,Excelente,NaN,407500000,92,92
427,427,2.0,1.0,4.0,9 a 15 años,15,$ 140.000 COP,"$ 3.928.571,43*m²",NaN,Excelente,NaN,165000000,42,42
428,428,3.0,2.0,4.0,16 a 30 años,3,$ 110.000 COP,"$ 5.266.666,67*m²",1.0,Excelente,NaN,395000000,75,75
429,429,3.0,3.0,5.0,más de 30 años,3,$ 150.000 COP,"$ 4.196.428,57*m²",1.0,Bueno,NaN,470000000,112,112


In [23]:
print("shape:", df_inmuebles.shape)
print("\ncolumns:", list(df_inmuebles.columns))
print("\ndtypes:")
print(df_inmuebles.dtypes)

shape: (430, 14)

columns: ['Unnamed: 0', 'Habitaciones', 'Baños', 'Estrato', 'Antigüedad', 'Piso N°', 'Administración', 'Precio m²', 'Parqueaderos', 'Estado', 'Tipo de apartamento', 'Precio', 'Área construída (m²)', 'Área privada (m²)']

dtypes:
Unnamed: 0                int64
Habitaciones            float64
Baños                   float64
Estrato                 float64
Antigüedad                  str
Piso N°                     str
Administración              str
Precio m²                   str
Parqueaderos            float64
Estado                      str
Tipo de apartamento         str
Precio                    int64
Área construída (m²)        str
Área privada (m²)           str
dtype: object


In [24]:
df_inmuebles.info()

<class 'pandas.DataFrame'>
RangeIndex: 430 entries, 0 to 429
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            430 non-null    int64  
 1   Habitaciones          272 non-null    float64
 2   Baños                 271 non-null    float64
 3   Estrato               309 non-null    float64
 4   Antigüedad            217 non-null    str    
 5   Piso N°               214 non-null    str    
 6   Administración        274 non-null    str    
 7   Precio m²             274 non-null    str    
 8   Parqueaderos          212 non-null    float64
 9   Estado                185 non-null    str    
 10  Tipo de apartamento   19 non-null     str    
 11  Precio                430 non-null    int64  
 12  Área construída (m²)  274 non-null    str    
 13  Área privada (m²)     274 non-null    str    
dtypes: float64(4), int64(2), str(8)
memory usage: 47.2 KB


In [25]:
df_inmuebles.describe()

,Unnamed: 0,Habitaciones,Baños,Estrato,Parqueaderos,Precio
count,430.000000,272.000000,271.000000,309.000000,212.000000,4.300000e+02
mean,214.500000,2.665441,2.383764,4.789644,1.457547,3.990034e+09
std,124.274562,1.375263,1.057755,1.030897,0.676687,7.231303e+10
min,0.000000,1.000000,1.000000,2.000000,1.000000,9.000000e+07
25%,107.250000,2.000000,2.000000,4.000000,1.000000,2.700000e+08
50%,214.500000,3.000000,2.000000,5.000000,1.000000,4.255000e+08
75%,321.750000,3.000000,3.000000,6.000000,2.000000,6.300000e+08
max,429.000000,20.000000,6.000000,6.000000,5.000000,1.500000e+12


In [26]:
df_inmuebles.describe(include='object')

,Antigüedad,Piso N°,Administración,Precio m²,Estado,Tipo de apartamento,Área construída (m²),Área privada (m²)
count,217,214,274,274,185,19,274,274
unique,5,17,129,189,5,3,117,85
top,1 a 8 años,Otro,No definida,"$ 4.032.258,06*m²",Excelente,Duplex,50,0
freq,78,53,75,7,113,9,11,115


**Características principales de Precios inmuebles Medellín**

- Tiene **430 inmuebles** y **14 columnas**. Es el dataset más pequeño.
- La primera columna, `Unnamed: 0`, es el índice que se guardó por error al exportar el archivo; no es un dato.
- Solo `Habitaciones`, `Baños`, `Estrato`, `Parqueaderos` y `Precio` son numéricas. `Administración`, `Precio m²` y las dos áreas quedaron como **texto** porque traen símbolos (`$`, `COP`, `*m²`, puntos de miles y espacios).
- Casi todas las columnas tienen muchos vacíos; `Tipo de apartamento` solo tiene 19 valores.
- `Precio` tiene un máximo de 1.500.000.000.000 pesos, un valor imposible frente a la mediana (unos 425 millones).
- **No tiene ninguna columna de ubicación** (ni barrio ni comuna).

### 3.5 Tabla de diagnóstico estructural

In [27]:
filas = []
for nombre, df in datasets.items():
    tipos = df.dtypes.astype(str).value_counts()
    filas.append({
        "Dataset": nombre,
        "Filas": df.shape[0],
        "Columnas": df.shape[1],
        "Tipos de datos": ", ".join(f"{t} ({c})" for t, c in tipos.items()),
        "Valores nulos": int(df.isnull().sum().sum()),
    })

pd.DataFrame(filas)

,Dataset,Filas,Columnas,Tipos de datos,Valores nulos
0,Información Predios,1041413,15,"int64 (9), str (4), float64 (2)",5800
1,Información Lotes,308129,8,"int64 (4), float64 (3), str (1)",0
2,Medellín Properties 2023,9999,12,"str (5), float64 (5), int64 (2)",6028
3,Precios inmuebles Medellín,430,14,"str (8), float64 (4), int64 (2)",2365


| Dataset | Filas | Columnas | Tipos de datos | Observaciones |
|---|---|---|---|---|
| Información Predios | 1.041.413 | 15 | 9 enteros, 2 decimales, 4 texto | `CBML` y `Matricula` leídos como números; `NroPropietarios` siempre 0; 5.800 nulos y 590.022 "--" en `Urbanizacion`; estrato 0 en 391.024 predios. |
| Información Lotes | 308.129 | 8 | 4 enteros, 3 decimales, 1 texto | Sin nulos; `CBML` como número y `ZonaGeo` como decimal; lotes con área 0 y con área construida o pisos incoherentes. |
| Medellín Properties 2023 | 9.999 | 12 | 2 enteros, 5 decimales, 5 texto | Coordenadas y `garages` como texto; vacíos en `age` y `garages`; 100 filas duplicadas; precios y áreas extremos. |
| Precios inmuebles Medellín | 430 | 14 | 2 enteros, 4 decimales, 8 texto | Columna de índice sobrante; montos y áreas como texto; muchos vacíos; sin columna de ubicación. |

## Momento 4. Diagnóstico inicial de calidad

En este momento **no se limpian los datos**: solo se identifican y se documentan los problemas.

### 4.1 Valores faltantes

In [28]:
for nombre, df in datasets.items():
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    print(f"== {nombre} ({df.shape[0]} filas)")
    if len(nulos) == 0:
        print("   Sin valores nulos")
    for columna, cantidad in nulos.items():
        print(f"   {columna:25} {cantidad:>6} nulos ({cantidad / df.shape[0] * 100:.1f} %)")

== Información Predios (1041413 filas)
   Urbanizacion                5800 nulos (0.6 %)
== Información Lotes (308129 filas)
   Sin valores nulos
== Medellín Properties 2023 (9999 filas)
   rooms                         63 nulos (0.6 %)
   baths                        122 nulos (1.2 %)
   age                         2637 nulos (26.4 %)
   garages                     3206 nulos (32.1 %)
== Precios inmuebles Medellín (430 filas)
   Habitaciones                 158 nulos (36.7 %)
   Baños                        159 nulos (37.0 %)
   Estrato                      121 nulos (28.1 %)
   Antigüedad                   213 nulos (49.5 %)
   Piso N°                      216 nulos (50.2 %)
   Administración               156 nulos (36.3 %)
   Precio m²                    156 nulos (36.3 %)
   Parqueaderos                 218 nulos (50.7 %)
   Estado                       245 nulos (57.0 %)
   Tipo de apartamento          411 nulos (95.6 %)
   Área construída (m²)         156 nulos (36.3 %)
   Área 

Además de los nulos, hay **valores de relleno** que en la práctica también son datos faltantes:

In [29]:
print("Predios con Urbanizacion = '--':", (df_predios["Urbanizacion"] == "--").sum())
print("Valores distintos en NroPropietarios:", df_predios["NroPropietarios"].unique().tolist())
print("Predios con Estrato = 0:", (df_predios["Estrato"] == 0).sum())
print("Propiedades con latitude = 'INVALID':", (df_propiedades["latitude"] == "INVALID").sum())
print("Propiedades con administration_price = 0:", (df_propiedades["administration_price"] == 0).sum())
print("Inmuebles con Administración = 'No definida':", (df_inmuebles["Administración"] == "No definida").sum())
print("Inmuebles con Piso N° = 'Otro':", (df_inmuebles["Piso N°"] == "Otro").sum())

columnas_sin_precio = df_inmuebles.drop(columns=["Unnamed: 0", "Precio"])
print("Inmuebles que solo tienen el precio (resto vacío):", columnas_sin_precio.isnull().all(axis=1).sum())

Predios con Urbanizacion = '--': 590022
Valores distintos en NroPropietarios: [0]
Predios con Estrato = 0: 391024
Propiedades con latitude = 'INVALID': 54
Propiedades con administration_price = 0: 3930
Inmuebles con Administración = 'No definida': 75
Inmuebles con Piso N° = 'Otro': 53
Inmuebles que solo tienen el precio (resto vacío): 121


**Cómo afectan el análisis:**

- En **Predios**, `Urbanizacion` está vacía o con "--" en 595.822 predios (57,2 %). No se puede usar el nombre de la urbanización para reconocer conjuntos. `NroPropietarios` no aporta nada porque siempre vale 0.
- El **estrato 0** (391.024 predios) probablemente corresponde a predios no residenciales (parqueaderos, locales, lotes); si se toma como un estrato real, los promedios por estrato quedan mal.
- En **Medellín Properties**, `age` falta en 2.637 inmuebles y `garages` en 3.206. Además, un `administration_price` de 0 en 3.930 inmuebles puede ser "no paga administración" o "no se informó": no se puede saber.
- En **Precios inmuebles**, 121 de 430 filas (28,1 %) solo tienen el precio. Esas filas no sirven para relacionar la administración con el estrato o el área.

### 4.2 Tipos de datos

**Código CBML.** El diccionario de datos dice que `CBML` es texto (`string`), pero pandas lo leyó como número. Al hacerlo se pierde el cero inicial de las comunas 1 a 9:

In [30]:
print("Tipo de CBML en predios:", df_predios["CBML"].dtype)
print("Tipo de CBML en lotes:  ", df_lotes["CBML"].dtype)

# Lectura de prueba de las primeras filas del archivo de lotes con CBML como texto
cbml_texto = pd.read_csv(RUTA_DATOS + "informacion_lotes.csv", dtype={"CBML": str}, nrows=3)["CBML"]
print("\nCBML como texto: ", list(cbml_texto))
print("CBML como número:", list(df_lotes["CBML"].head(3)))

Tipo de CBML en predios: int64
Tipo de CBML en lotes:   int64

CBML como texto:  ['01010020003', '01010020010', '01010020014']
CBML como número: [1010020003, 1010020010, 1010020014]


Con el cero, todos los CBML tienen 11 dígitos (comuna 2 + barrio 2 + manzana 3 + lote 4). Como número, los de las comunas 1 a 9 quedan con 10 dígitos y ya no se pueden partir en sus partes. En predios esto afecta a 353.005 registros (33,9 %).

**Otras columnas con tipo incorrecto:**

In [31]:
print("Lotes - ZonaGeo:", df_lotes["ZonaGeo"].dtype, "->", df_lotes["ZonaGeo"].head(2).tolist())
print("Propiedades - latitude:", df_propiedades["latitude"].dtype, "| longitude:", df_propiedades["longitude"].dtype)
print("Propiedades - garages:", df_propiedades["garages"].dtype, "->", df_propiedades["garages"].dropna().unique().tolist())
print("Propiedades - age (código, no años):", sorted(df_propiedades["age"].dropna().unique().tolist()))
print("Inmuebles - Administración:", df_inmuebles["Administración"].dropna().unique()[:4].tolist())
print("Inmuebles - Precio m²:", df_inmuebles["Precio m²"].dropna().unique()[:3].tolist())
print("Inmuebles - Área construída (m²):", df_inmuebles["Área construída (m²)"].dropna().unique()[:4].tolist())
print("Inmuebles - Estrato:", df_inmuebles["Estrato"].dtype, "(entero con vacíos -> decimal)")

Lotes - ZonaGeo: float64 -> [122121422212440.0, 122121422212440.0]
Propiedades - latitude: str | longitude: str
Propiedades - garages: str -> ['1', '2', '7', '3', '4', '5', '10', '6', '8', 'Más de 10', '9']
Propiedades - age (código, no años): [0.0, 1.0, 2.0, 3.0, 4.0]
Inmuebles - Administración: ['No definida', '$ 167.000 COP', '$ 180.000 COP', '$ 650.000 COP']
Inmuebles - Precio m²: ['$ 4.032.258,06*m²', '$ 6.428.571,43*m²', '$ 4.700.000*m²']
Inmuebles - Área construída (m²): ['62 ', '105 ', '50 ', '32 ']
Inmuebles - Estrato: float64 (entero con vacíos -> decimal)


| Dataset | Columna | Tipo actual | Tipo esperado | Causa |
|---|---|---|---|---|
| Predios y Lotes | `CBML` | `int64` | texto | Pandas lo detecta como número y quita el cero inicial |
| Predios | `Matricula`, `UsoPredial` | `int64` | texto o categoría | Son códigos; no tiene sentido sumarlos ni promediarlos |
| Lotes | `ZonaGeo` | `float64` | texto | Código largo convertido a decimal |
| Medellín Properties | `latitude`, `longitude` | texto | decimal | Valores "INVALID" |
| Medellín Properties | `garages` | texto | entero | Valor "Más de 10" |
| Medellín Properties | `age` | decimal | categoría | Es un código de rango de antigüedad (0 a 4) |
| Precios inmuebles | `Administración`, `Precio m²`, áreas | texto | decimal | Símbolos `$`, `COP`, `*m²`, puntos de miles, comas decimales y espacios |
| Precios inmuebles | `Estrato`, `Habitaciones`, `Baños` | decimal | entero | Tienen vacíos, y los vacíos obligan a usar decimal |

### 4.3 Posibles inconsistencias

**a) Los nombres de las columnas no coinciden con el diccionario de datos de MEData:**

In [32]:
import json

for archivo, df in [("Informacion_Predios.json", df_predios), ("Informacion_Lotes.json", df_lotes)]:
    with open(RUTA_DATOS + archivo, encoding="utf-8") as f:
        diccionario = [campo["name"] for campo in json.load(f)["fields"]]
    print(archivo)
    print("   En el diccionario y no en el CSV:", [c for c in diccionario if c not in df.columns])
    print("   En el CSV y no en el diccionario:", [c for c in df.columns if c not in diccionario])

Informacion_Predios.json
   En el diccionario y no en el CSV: ['Uso Predial', 'NomComuna', 'DS_Barrio']
   En el CSV y no en el diccionario: ['UsoPredial', 'nomComuna', 'DS_BARRIO']
Informacion_Lotes.json
   En el diccionario y no en el CSV: ['NomComuna', 'Zona Geo']
   En el CSV y no en el diccionario: ['nomComuna', 'ZonaGeo']


**b) Nombres de comunas mal escritos:**

In [33]:
print(sorted(df_predios["nomComuna"].unique().tolist()))

['ALTAVISTA', 'ARANJUEZ', 'BELEN', 'BUENOS AIRES', 'CASTILLA', 'DOCE DE OCTUBRE', 'EL POBLADO', 'GUAYABAL', 'LA AMERIA', 'LA CANDELARIA', 'LAURELES', 'MANRRIQUE', 'PALMITAS', 'POPULAR', 'ROBLEDO', 'SAN ANTONIO DE PRADO', 'SAN CRISTOBAL', 'SAN JAVIER', 'SANTA CRUZ', 'SANTA ELENA', 'VILLA HERMOSA']


Aparecen "MANRRIQUE" (debería ser Manrique) y "LA AMERIA" (debería ser La América). Si se cruzan estos datos con otra fuente por el nombre de la comuna, esas dos comunas no van a coincidir.

**c) El porcentaje de copropiedad de cada lote debería sumar 100:**

In [34]:
suma_por_lote = df_predios.groupby("CBML")["PorDes"].sum()
lotes_que_no_suman_100 = suma_por_lote[(suma_por_lote - 100).abs() > 1]

print("Lotes en total:", suma_por_lote.shape[0])
print("Lotes cuya suma de PorDes no es 100 (±1):", lotes_que_no_suman_100.shape[0])
print("Suma mínima y máxima encontradas:", suma_por_lote.min(), "y", suma_por_lote.max())
print("Predios con PorDes = 0:", (df_predios["PorDes"] == 0).sum())

Lotes en total: 308129
Lotes cuya suma de PorDes no es 100 (±1): 430
Suma mínima y máxima encontradas: 0.0 y 7400.0
Predios con PorDes = 0: 16868


**d) Lotes con áreas o pisos incoherentes:**

In [35]:
print("Lotes con AreaLote = 0:", (df_lotes["AreaLote"] == 0).sum())
print("Lotes sin área construida pero con pisos:", ((df_lotes["AreaCons"] == 0) & (df_lotes["numPisos"] > 0)).sum())
print("Lotes con área construida pero 0 pisos:", ((df_lotes["AreaCons"] > 0) & (df_lotes["numPisos"] == 0)).sum())

Lotes con AreaLote = 0: 80
Lotes sin área construida pero con pisos: 696
Lotes con área construida pero 0 pisos: 415


**e) Duplicados, coordenadas y valores extremos en los datasets de Kaggle:**

In [36]:
latitud = pd.to_numeric(df_propiedades["latitude"], errors="coerce")
longitud = pd.to_numeric(df_propiedades["longitude"], errors="coerce")
fuera_de_medellin = (latitud < 6.1) | (latitud > 6.4) | (longitud < -75.75) | (longitud > -75.45)

print("Medellín Properties")
print("   Filas duplicadas:", df_propiedades.duplicated().sum())
print("   Coordenadas fuera del área de Medellín:", fuera_de_medellin.sum())
print("   Rango de latitud:", latitud.min(), "a", latitud.max())
print("   Inmuebles con area = 0:", (df_propiedades["area"] == 0).sum())
print("   Inmuebles con area > 2.000 m²:", (df_propiedades["area"] > 2000).sum())
print("   Inmuebles con precio > 100.000 millones:", (df_propiedades["price"] > 1e11).sum())
print("   Inmuebles con precio < 50 millones:", (df_propiedades["price"] < 5e7).sum())
print("   Inmuebles con stratum = 0:", (df_propiedades["stratum"] == 0).sum())

print("\nPrecios inmuebles Medellín")
print("   Filas duplicadas (sin contar el índice):", df_inmuebles.drop(columns="Unnamed: 0").duplicated().sum())
print("   Precio máximo:", df_inmuebles["Precio"].max())

Medellín Properties
   Filas duplicadas: 100
   Coordenadas fuera del área de Medellín: 104
   Rango de latitud: -34.4663154 a 52.44976630000001
   Inmuebles con area = 0: 4
   Inmuebles con area > 2.000 m²: 21
   Inmuebles con precio > 100.000 millones: 11
   Inmuebles con precio < 50 millones: 4
   Inmuebles con stratum = 0: 5

Precios inmuebles Medellín
   Filas duplicadas (sin contar el índice): 185
   Precio máximo: 1500000000000


Las coordenadas se compararon con un rectángulo aproximado que encierra a Medellín (latitud 6,1 a 6,4 y longitud −75,75 a −75,45). 104 inmuebles quedan por fuera; algunos tienen latitudes negativas o mayores que 50, que corresponden a otros países. Los duplicados de *Precios inmuebles* son sobre todo filas que solo tienen el precio.

### 4.4 Limitaciones para responder la pregunta analítica

In [37]:
predios_por_lote = df_predios.groupby("CBML")["Matricula"].transform("count")
posible_ph = (df_predios["PorDes"] < 100) & (predios_por_lote > 1)

print("Predios con PorDes < 100 en lotes compartidos:", posible_ph.sum(),
      f"({posible_ph.sum() / df_predios.shape[0] * 100:.1f} %)")
print("Códigos distintos de UsoPredial:", sorted(df_predios["UsoPredial"].unique().tolist()))
print("Predios rurales (IndicadorU/R = 'R'):", (df_predios["IndicadorU/R"] == "R").sum())
print("Lotes de predios que existen en el archivo de lotes:", f"{df_predios['CBML'].isin(df_lotes['CBML']).mean() * 100:.1f} %")
print("Lotes del archivo de lotes que tienen predios:", f"{df_lotes['CBML'].isin(df_predios['CBML']).mean() * 100:.1f} %")

Predios con PorDes < 100 en lotes compartidos: 820173 (78.8 %)
Códigos distintos de UsoPredial: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 17]
Predios rurales (IndicadorU/R = 'R'): 19716
Lotes de predios que existen en el archivo de lotes: 100.0 %
Lotes del archivo de lotes que tienen predios: 100.0 %


1. **No hay una columna que diga si un predio está en propiedad horizontal.** El criterio propuesto en la EV02 (porcentaje de copropiedad menor que 100 en un lote compartido) marca 820.173 predios, el 78,8 % del total. Es un número muy alto porque incluye también parqueaderos, cuartos útiles y locales dentro de edificios. El criterio se debe validar con una muestra de fichas catastrales.
2. **`UsoPredial` viene en códigos (11 valores distintos) y no hay tabla que explique qué significa cada uno.** Sin esa tabla no se puede separar con seguridad lo habitacional de lo comercial, que es parte de la pregunta.
3. **Los datos de MEData son de 2021** (última actualización registrada en el portal). Describen la situación de ese año, no la actual.
4. **Los datasets de Kaggle no tienen origen documentado** y son avisos de venta, no un registro de predios. *Medellín Properties* incluye coordenadas fuera de Medellín y tiene barrio, pero no comuna. *Precios inmuebles* no tiene ninguna ubicación, así que no se puede relacionar con las comunas.
5. **Predios y Lotes sí se pueden unir:** todos los `CBML` de un archivo existen en el otro, así que el área y los pisos del lote se pueden asociar a sus predios.
6. **Los corregimientos** (códigos de comuna 50 a 90) están mezclados con las comunas urbanas; la pregunta habla de comunas, así que hay que decidir si se incluyen.

### 4.5 Tabla de hallazgos y acciones de limpieza propuestas

| Dataset | Problema identificado | Evidencia encontrada | Posible impacto | Acción propuesta |
|---|---|---|---|---|
| Predios | `CBML` leído como número | `dtype` `int64`; 353.005 códigos pierden el cero inicial | No se puede partir el código ni unir bien con otras fuentes que lo usen como texto | Leer con `dtype={"CBML": str}` |
| Predios | Columna sin información | `NroPropietarios` siempre vale 0 | Ruido en el análisis | Eliminar la columna |
| Predios | Valores de relleno en `Urbanizacion` | 5.800 nulos y 590.022 "--" | No sirve para reconocer conjuntos por nombre | Reemplazar "--" por nulo y no usarla como criterio |
| Predios | Estrato 0 | 391.024 predios | Distorsiona promedios por estrato | Tratar 0 como "sin estrato" (nulo) o separarlo |
| Predios | Nombres de comuna mal escritos | "MANRRIQUE", "LA AMERIA" | Fallan los cruces por nombre | Corregir con un diccionario de reemplazos |
| Predios | Suma de copropiedad distinta de 100 | 430 lotes; suma máxima de 7.400 | El criterio de propiedad horizontal puede fallar en esos lotes | Revisar y marcar esos lotes; excluirlos o validarlos |
| Predios | No hay marca de propiedad horizontal ni tabla de usos | 11 códigos de `UsoPredial` sin descripción | No se puede filtrar lo habitacional ni lo que es PH con certeza | Pedir la tabla de códigos a Catastro y validar el criterio con fichas |
| Predios y Lotes | Nombres de columnas distintos al diccionario | `UsoPredial` vs. `Uso Predial`, `DS_BARRIO` vs. `DS_Barrio`, `ZonaGeo` vs. `Zona Geo`… | Errores al escribir código a partir del diccionario | Renombrar columnas con un estándar único |
| Lotes | `ZonaGeo` como decimal | Valores como 1.22e+14 | Se pueden perder dígitos del código | Leer como texto |
| Lotes | Áreas y pisos incoherentes | 80 lotes con área 0; 696 sin área construida pero con pisos; 415 con área construida y 0 pisos | Estimaciones erróneas de edificios | Marcar como inconsistentes y revisarlos |
| Medellín Properties | Coordenadas como texto | 54 filas con "INVALID" | No se pueden ubicar en un mapa | Convertir con `pd.to_numeric(errors="coerce")` |
| Medellín Properties | Coordenadas fuera de Medellín | 104 inmuebles | Se mezclan datos de otros municipios o países | Filtrar por el rango de Medellín |
| Medellín Properties | Duplicados | 100 filas repetidas | Se cuentan inmuebles dos veces | `drop_duplicates()` |
| Medellín Properties | Valores extremos | Áreas de 0 y de más de 2.000 m²; 11 precios mayores de 100.000 millones | Promedios inflados | Definir rangos válidos y excluir o revisar los extremos |
| Medellín Properties | `garages` texto y `age` como código | "Más de 10"; `age` de 0 a 4 | Cálculos incorrectos | Convertir `garages` a número y `age` a categoría |
| Precios inmuebles | Columna de índice sobrante | `Unnamed: 0` | Ruido | Leer con `index_col=0` |
| Precios inmuebles | Montos y áreas como texto | "$ 167.000 COP", "$ 4.032.258,06*m²", "62 " | No se pueden hacer cálculos | Quitar símbolos y espacios y convertir a número |
| Precios inmuebles | Filas casi vacías y duplicadas | 121 filas solo con precio; 185 duplicadas sin índice | Muestra útil muy pequeña | Eliminar filas sin datos además del precio |
| Precios inmuebles | Sin ubicación | No existe columna de barrio ni comuna | No responde la pregunta por comunas | Usarlo solo como apoyo (administración vs. estrato) o descartarlo |

### Conclusión del diagnóstico

Los datasets de MEData (Predios y Lotes) son los adecuados para responder la pregunta: cubren toda la ciudad, se relacionan por el `CBML` y casi no tienen nulos. Sus problemas principales son de **tipos de datos** (códigos leídos como números), **valores de relleno**, **nombres mal escritos** y, sobre todo, la **falta de una marca directa de propiedad horizontal y de una tabla de usos prediales**. Los datasets de Kaggle tienen más problemas de calidad (tipos incorrectos, duplicados, extremos y vacíos) y no tienen comuna, así que solo servirían como apoyo. Antes de analizar hay que aplicar las acciones de la tabla anterior y validar el criterio para identificar la propiedad horizontal.